In [32]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import warnings
warnings.simplefilter('ignore')

In [ ]:
path = 'dataset/Crop_recommendation.csv'
df = pd.read_csv(path)
print(df.shape)
df.head()

In [ ]:
df.describe()

In [35]:
# doing 60-40 split
TEST_SIZE = 0.4

features = df[['N', 'P', 'K', 'temperature', 'humidity']]#, 'ph', 'rainfall']]
features.rename(columns={"humidity": "moisture"}, inplace=True)
target = df['label']
Xtrain, Xtest, Ytrain, Ytest = train_test_split(features, target, test_size=TEST_SIZE)

In [36]:

analysis = {}


def evaluate_model(model, name, indepth):
    print('-' * 20, name, '-' * 20)
    y_pred = model.predict(Xtest)

    # gives the precision, f1 score and recall for each class
    if indepth:
        print(classification_report(Ytest, y_pred))
    
    # gives the accuracy score
    test_accuracy = accuracy_score(Ytest, y_pred)
    train_accuracy = accuracy_score(Ytrain, model.predict(Xtrain))
    print(f'Test Accuracy: {test_accuracy*100:.3f}%')
    print(f'Train Accuracy: {train_accuracy*100:.3f}%')

    # plots the confusion matrix as a heatmap
    # results are good if the colors are in a diagonal line
    mat = confusion_matrix(Ytest, y_pred)
    sns.heatmap(mat, annot=True, fmt='d', cbar=False, yticklabels=model.classes_, xticklabels=model.classes_)

    # storing data to compare later

    analysis[name] = {
        'name': name,
        'accuracy': test_accuracy,
        'model': model,
    }

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'Decision Tree', indepth=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'Random Forest', indepth=False)

In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'Naive Bayes', indepth=False)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'KNN', indepth=False)

In [ ]:
from sklearn.svm import SVC

model = SVC()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'SVM', indepth=False)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'Logistic Regression', indepth=False)

In [ ]:
from sklearn.linear_model import Perceptron

model = Perceptron()
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'Perceptron', indepth=False)

In [ ]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(15, 15))
model.fit(Xtrain, Ytrain)
evaluate_model(model, 'MLP', indepth=False)

In [ ]:
from matplotlib import pyplot as plt

plot_data = {
    'name': [],
    'accuracy': [],
}

for name, data in analysis.items():
    plot_data['name'].append(data['name'])
    plot_data['accuracy'].append(data['accuracy'])


sns.barplot(
    data=plot_data,
    x='accuracy', y='name',
    palette='dark',
    orient='h',
    hue='accuracy',
    dodge=False,
    legend=False,
)
plt.xlim(0.6, 1.0)

In [ ]:
best_model_name = max(analysis, key=lambda x: analysis[x]['accuracy'])
best_model_data = analysis[best_model_name]

print("-"*18, f"BEST MODEL: {best_model_data['accuracy'] * 100:.2f}%", "-"*18)
evaluate_model(best_model_data['model'], best_model_name, indepth=True)

import pickle
pickle.dump(best_model_data['model'], open(f"models/{best_model_name.replace(' ', '-')}--model.pkl", 'wb'))